# 02 – Prepare replay source
Moves full extracts of transactional tables from landing to the simulated source volume
and resets their bronze tables and checkpoints, so monthly files can be replayed.

**Destructive for bronze tables listed in `REPLAY_TABLES`.** Run once before the first replay.

In [0]:
dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
CATALOG = dbutils.widgets.get("catalog")

LANDING = f"/Volumes/{CATALOG}/landing/raw/olist"
SOURCE = f"/Volumes/{CATALOG}/landing/source/olist"
CHECKPOINTS = f"/Volumes/{CATALOG}/bronze/checkpoints/olist"

# Transactional tables that arrive monthly; reference tables stay as one file
REPLAY_TABLES = ["orders", "order_items", "order_payments", "order_reviews"]

for table in REPLAY_TABLES:
    landing_dir = f"{LANDING}/{table}"
    source_dir = f"{SOURCE}/{table}"
    dbutils.fs.mkdirs(source_dir)

    # 1) Move full extract to source (skip if already moved -> safe to re-run)
    for f in dbutils.fs.ls(landing_dir):
        if f.name.startswith("olist_"):
            dbutils.fs.mv(f.path, f"{source_dir}/{f.name}")
            print(f"MOVED    {f.name} -> source/{table}/")

    # 2) Drop bronze table and 3) its checkpoint, so ingestion starts from scratch
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.bronze.olist_{table}")
    dbutils.fs.rm(f"{CHECKPOINTS}/{table}", recurse=True)
    print(f"RESET    bronze.olist_{table} + checkpoint")